# Project V: Kaggle training session
Runs several `(exp, seed)` trainings as parallel subprocesses (1 torch thread each).
Outputs go to `/kaggle/working/results`. To resume, attach a previous session's output as
input (`kernel_sources`). It is copied into `results/`, and every run starts with
`--resume auto`. Logic lives in `projectv`; this notebook only launches it.

In [ ]:
# PARAMETERS (generated by scripts/make_kaggle_kernels.py)
RUNS = [("reach_dr", 0)]
DEVICE = "cpu"
SHA = "main"
MAX_HOURS = 11.0
ALGO = ""
EXTRA = ""
RESUME_REQUIRED = False


In [ ]:
import glob, os, shutil, subprocess, sys, time
REPO = "https://github.com/Darsh-14/Project-v.git"
SRC, WORK = "/kaggle/tmp/pv", "/kaggle/working"
RES, LOGS = f"{WORK}/results", f"{WORK}/logs"
os.makedirs(LOGS, exist_ok=True)
def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)
sh(f"rm -rf {SRC} && git clone -q {REPO} {SRC} && cd {SRC} && git checkout -q {SHA} && git log -1 --oneline")
sh(f"cd {SRC} && pip install -q -r requirements.lock && pip install -q -e . --no-deps")
sh("python --version; nproc; free -g | head -2; nvidia-smi -L || true")

In [ ]:
# Resume: merge outputs of attached previous sessions into results/ (newest step files win).
# attached kernel outputs are mounted at varying depths (e.g. /kaggle/input/notebooks/<user>/<slug>/)
for prev in sorted({os.path.dirname(c) for c in glob.glob("/kaggle/input/**/results/ckpt", recursive=True)}):
    print("resume input:", prev)
    shutil.copytree(prev, RES, dirs_exist_ok=True)
sh(f"ls -R {RES}/ckpt 2>/dev/null | head -50 || true")
if RESUME_REQUIRED:  # fail fast instead of silently restarting from step 0
    for exp, seed in RUNS:
        d = f"{RES}/ckpt/{exp}_s{seed}"
        assert os.path.isdir(d) and any(f.startswith("step_") for f in os.listdir(d)), (
            f"resume input missing for {exp}_s{seed}: {d}; inputs: {glob.glob('/kaggle/input/*')}")
    print("resume inputs verified for", RUNS)


In [ ]:
import torch
n_gpu = torch.cuda.device_count()
procs = {}
for i, (exp, seed) in enumerate(RUNS):
    dev = "cpu" if DEVICE == "cpu" else f"cuda:{i % max(n_gpu, 1)}"
    cmd = [sys.executable, "-m", "projectv.train.train", "--exp", f"configs/exp/{exp}.yaml",
           "--seed", str(seed), "--resume", "auto", "--out", RES, "--threads", "1",
           "--device", dev, "--max-wall-hours", str(MAX_HOURS), "--no-wandb"]
    if ALGO: cmd += ["--algo", ALGO]
    if EXTRA: cmd += EXTRA.split()
    env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
    log = open(f"{LOGS}/{exp}_s{seed}.log", "a")
    procs[(exp, seed)] = subprocess.Popen(cmd, cwd=SRC, env=env, stdout=log, stderr=subprocess.STDOUT)
    print("launched", exp, seed, dev, flush=True)

In [ ]:
# Monitor until all runs exit (each stops itself before MAX_HOURS and saves model + replay).
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    print(f"--- {(time.time() - t0) / 3600:.2f} h", flush=True)
    for (exp, seed), p in procs.items():
        csvs = glob.glob(f"{RES}/train_logs/{exp}_s{seed}*/eval.csv")
        last = open(csvs[0]).read().strip().splitlines()[-1] if csvs else "no checkpoint yet"
        print(f"{exp}_s{seed} rc={p.poll()} | {last}", flush=True)
print({k: p.returncode for k, p in procs.items()})

In [ ]:
sh(f"du -sh {RES}/* || true")
for f in sorted(glob.glob(f"{RES}/train_logs/*/eval.csv")):
    print(f, open(f).read().strip().splitlines()[-1])
for f in sorted(glob.glob(f"{LOGS}/*.log")):
    print("==", f); print("".join(open(f).readlines()[-5:]))